# Configurable linear plate

Standalone run on the supplied **SWG 256×256 geometry, 12 µm × 12 µm**. Edit the **Parameters** cell, then select **Kernel → Restart Kernel and Run All Cells**. Paths are relative to the repository root. Results are saved under `outputs/`.

No trained weights or external dataset are required. The input geometry is already included.


In [ ]:
%matplotlib inline
from pathlib import Path
import os
import sys

root_candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in root_candidates if (p / "plate_models.py").is_file()
             and (p / "data/swg_256.npz").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or its notebooks directory")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)

from pathlib import Path
import numpy as np
from plate_models import PlateModel


## Parameters

Change the input file/key, physical domain size, physical parameters and output path here. Support maps use 1 for supports and 0 for suspended film. `width_um` sets **both** length and width of the square domain; independent unequal x/y extents are not exposed by these solvers. Changing the domain size scales every feature; change the input pixels to change individual teeth or gaps.
Mechanical units: width in µm, pressure and Young's modulus in Pa, thickness in m, tension in N/m, prestrain dimensionless.


In [ ]:
structure_file = "data/swg_256.npz"
structure_key = "array"
width_um = 12.0
q_pa = -33.87
young_pa = 167.3e9
poisson = 0.19
film_thickness_m = 1e-9
tension_n_per_m = 0.0
output_file = "outputs/swg_linear.npz"


## Calculate and save

The result dictionary contains `w` in µm, dimensionless `strain_bend`, dimensionless `strain_geom` and convergence `info`. The example converts deflection to metres when saving it. Supports and outer boundaries have zero deflection. Optional `H_floor` specifies floor depth in µm; `kc` controls penalty contact. At the listed parameters the SWG bending peak is again approximately 0.230908%.

In [ ]:
with np.load(structure_file) as data:
    support = data[structure_key]
model = PlateModel(support, width_um=width_um,
                   E=young_pa, nu=poisson, h=film_thickness_m)
result = model.solve_linear(q_pa=q_pa, T=tension_n_per_m)
if not result["info"]["converged"]:
    raise RuntimeError("Linear solve did not converge")
Path(output_file).parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(output_file, support_map=support, strain_fraction=result["strain_bend"],
                    deflection_m=result["w"] * 1e-6, width_m=width_um * 1e-6)
print("Peak bending strain (%):", result["strain_bend"].max() * 100)


## Inspect the result

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].imshow(support, origin="lower", extent=[0, width_um, 0, width_um])
axes[0].set_title("SWG supports")
im = axes[1].imshow(result["strain_bend"] * 100, origin="lower", extent=[0, width_um, 0, width_um])
axes[1].set_title("Strain (%)")
fig.colorbar(im, ax=axes[1])
for ax in axes:
    ax.set_xlabel("x (um)")
    ax.set_ylabel("y (um)")
plt.show()
